In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
os.chdir('/Users/jinghuang/Desktop/SSTP/Parse')

In [ ]:
def flatten_labels(label_tree, blanks=True):
    flat = []
    for category, tests in label_tree.items():
        if blanks: 
            flat.append((category,'', ''))
        else:
            flat.append((category,))
        if isinstance(tests, list):
            for t in tests:
                if blanks:
                    flat.append((category, t, ''))
                else:
                    flat.append((category, t))
        else:                       # dict
            for test_name, subs in tests.items():
                if subs:
                    for sub in subs:
                        flat.append((category, test_name, sub))
                else:
                    if blanks:
                        flat.append((category, test_name, ''))
                    else:
                        flat.append((category, test_name))
    return flat

def print_aligned(rows, aligns=None, spacing=1):
    """
    Print rows (iterables) with comma separators and aligned columns.

    rows   : iterable of row iterables (e.g., list of tuples/lists)
    aligns : iterable of alignment chars per column: '<' (left), '>' (right), '^' (center).
             If None, all columns left-aligned.
    spacing: number of spaces after each comma.
    """
    # Convert everything to strings first
    str_rows = [[str(cell) for cell in row] for row in rows]

    # Determine max width per column
    widths = [max(len(cell) for cell in col) for col in zip(*str_rows)]

    # Default alignments
    if aligns is None:
        aligns = ['<'] * len(widths)
    else:
        # pad/truncate aligns to match number of columns
        aligns = (list(aligns) + ['<'] * len(widths))[:len(widths)]

    # Build a format string segment for each column
    col_specs = [
        f'{{:{a}{w}}}'  # e.g., '{:<10}' or '{:>5}'
        for a, w in zip(aligns, widths)
    ]

    # Join with comma + spacing spaces
    sep = ', ' * spacing if spacing else ','
    fmt = sep.join(col_specs)

    # Print each row
    for row in str_rows:
        print(fmt.format(*row))

In [ ]:
# Get patient
patient = Path('case_outputs/tables_clean/00000.csv')
patient = pd.read_csv(patient)

# Get field_labels
fl_path = Path('instructions/field_labels.json')
fl_flat = [(field, False) for field in flatten_labels(json.loads(fl_path.read_text()), blanks=False)]

patient

In [ ]:
idx1, idx2 = 0,0
res = []
printstream = []
while idx1 < patient.shape[0]-1:
    # See if all fields have been checked
    if idx2 < len(fl_flat):
        Y, used = fl_flat[idx2]
        
        # If the field is already filled out
        if used: 
            idx2 += 1
            continue
        
        # See if field is a category or test
        if len(Y) == 1: # if category, needs to also match casewise (uppercase)
            X = patient.iloc[idx1]['test_info']
            Y0 = Y[0].upper()
            flag = 'CATEG'
        else:           # if tname, just check with lower
            X = patient.iloc[idx1]['test_info'].lower()
            Y0 = Y[-1].lower()
            flag = 'TNAME'
        
        # Check if field is in the patient row
        if Y0 in X:
            res.append((idx2, idx1))
            printstream.append((flag, f"{idx1:02d}", f"{idx2:02d}", Y, patient.iloc[idx1].values, f"'{Y0}' is in '{X}'"))
            fl_flat[idx2] = Y, True
            idx1 += 1
            idx2 = 0
        else:  # if field not in, then next field
            idx2 += 1
    else: # if so, the patient row does not correspond
        res.append((idx2, None))
        printstream.append(("NMTCH", f"{idx1:02d}", f"{idx2:02d}", '', patient.iloc[idx1].values, ''))
        idx1 += 1
        idx2 = 0
print_aligned(printstream)

In [ ]:
"""fl_flat = flatten_labels(json.loads(fl_path.read_text()))
multi_cols = pd.MultiIndex.from_tuples(
    fl_flat, names=["Category", "Test", "Subtest"]
)
data = np.empty((3, len(multi_cols)), dtype=object)
patient_no = 0
for test_name_idx, patient_row_idx in res:
    if patient_row_idx is not None:
        data[patient_no][test_name_idx] = patient.iloc[patient_row_idx]['Middle']
        data[patient_no+1][test_name_idx] = patient.iloc[patient_row_idx]['Left']
        data[patient_no+2][test_name_idx] = patient.iloc[patient_row_idx]['Right']

df = pd.DataFrame(data, columns=multi_cols)

file_name = 'report_with_multi_header.xlsx'

# Simply remove 'index=False'. This resolves the error.
with pd.ExcelWriter(file_name, engine="xlsxwriter") as writer:
    # dump the DataFrame
    df.to_excel(writer, sheet_name="Results")

    # grab the native XlsxWriter worksheet object
    ws = writer.sheets["Results"]

    ws.set_column(first_col=0,
                  last_col=len(df.columns) - 1,
                  width=20)


print(f"✅ Excel file '{file_name}' was created successfully (with index column).")"""

# ----------------------------------  SET-UP  ----------------------------------
fl_flat = flatten_labels(json.loads(fl_path.read_text()))
multi_cols = pd.MultiIndex.from_tuples(fl_flat,
                                       names=["Category", "Test", "Subtest"])

# ---- identify your patients ---------------------------------------------------
# Example: one CSV per patient in a list; adapt to your real source
patients = [patient]                      # <- replace with real list/iterator
patient_ids = [p.attrs.get("id", i) for i, p in enumerate(patients)]

# ---- allocate the backing array ----------------------------------------------
n_patients = len(patient_ids)
data = np.empty((n_patients * 3, len(multi_cols)), dtype=object)

# ---- fill it ------------------------------------------------------------------
for p_idx, p_df in enumerate(patients):
    base = p_idx * 3                      # first row (raw_score) for this patient
    for test_name_idx, patient_row_idx in res:
        if patient_row_idx is None:
            continue
        data[base    , test_name_idx] = p_df.iloc[patient_row_idx]["raw_score"]  # raw_score
        data[base + 1, test_name_idx] = p_df.iloc[patient_row_idx]["comments"]    # comments
        data[base + 2, test_name_idx] = p_df.iloc[patient_row_idx]["test_info"]   # test_info

# ---- build the row MultiIndex -------------------------------------------------
row_tuples = [
    (pid, label)
    for pid in patient_ids
    for label in ("raw_score", "comments", "test_info")
]
row_index = pd.MultiIndex.from_tuples(row_tuples,
                                      names=["Patient", "Field"])

df = pd.DataFrame(data, index=row_index, columns=multi_cols)

# ----------------------------------  EXPORT  ----------------------------------
file_name = "report_with_multi_header.xlsx"

with pd.ExcelWriter(file_name, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Results")          # keep the index; it's meaningful

    ws = writer.sheets["Results"]
    ws.set_column(first_col=0,
                  last_col=len(df.columns) - 1,
                  width=12)                            # uniform width (or auto-fit loop)

print(f"✅ Excel file '{file_name}' was created successfully.")
